# Frame extraction and tactical visualization
This notebook is split from the original thesis workflow to keep each stage focused and easier to maintain.


In [ ]:
# NOTE: comment translated/cleaned for English-only repository.
from pathlib import Path
import os
_here = Path.cwd().resolve()
if _here.name == "notebooks":
    os.chdir(_here.parent)
elif not (_here / "image").is_dir() and (_here.parent / "image").is_dir():
    os.chdir(_here.parent)
print("Project root:", Path.cwd())


In [ ]:
def extract_frame_players_and_roles(coordinates_df, row):
    timestamp = row['timestamp']
    passer_id = row['passedPlayerId']
    team_id = row['passedPlayerTeamId']

    frame = coordinates_df[coordinates_df['timestamp'] == timestamp]
    if frame.empty or pd.isna(timestamp):
        raise ValueError(f"No coordinate data found for timestamp {timestamp}")
    frame = frame.iloc[0]

    # Extract all players with valid IDs and positions
    player_rows = []
    for i in range(40):
        pid = frame.get(f'playerId{i}', -100)
        x = frame.get(f'posX{i}', -100)
        y = frame.get(f'posY{i}', -100)
        if pid == -100 or x == -100 or y == -100:
            continue
        player_rows.append({'player_index': i, 'player_id': pid, 'posX': x, 'posY': y})

    df_players = pd.DataFrame(player_rows)

    # Retrieve teammate and opponent IDs from pass record
    teammate_ids = [row.get(f'teamMatePlayerId{i}') for i in range(1, 11)]
    opponent_ids = [row.get(f'opponentPlayerId{i}') for i in range(1, 12)]

    # Assign roles
    df_players['is_passer'] = df_players['player_id'] == passer_id
    df_players['is_teammate'] = df_players['player_id'].isin(teammate_ids)
    df_players['is_opponent'] = df_players['player_id'].isin(opponent_ids)

    return row, df_players


In [ ]:
def draw_frame_players(df_players, pass_row):
    # Set up pitch display
    plt.figure(figsize=(10, 6.5))
    plt.xlim(0, 105)
    plt.ylim(0, 68)
    plt.gca().set_facecolor('lightgreen')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

    receiver_id = pass_row.get('receivedPlayerId', None)

    # Draw passer
    passer = df_players[df_players['is_passer']]
    plt.scatter(passer['posX'], passer['posY'], color='blue', s=100, label='Passer')

    # Draw teammates (excluding receiver)
    teammates = df_players[df_players['is_teammate'] & (df_players['player_id'] != receiver_id)]
    plt.scatter(teammates['posX'], teammates['posY'], color='white', s=60, label='Teammates')

    # Draw receiver
    receiver = df_players[df_players['player_id'] == receiver_id]
    if not receiver.empty:
        plt.scatter(receiver['posX'], receiver['posY'], color='yellow', edgecolors='black', s=100, label='Receiver', zorder=5)

    # Draw opponents
    opponents = df_players[df_players['is_opponent']]
    plt.scatter(opponents['posX'], opponents['posY'], color='black', s=60, label='Opponents')



    # Draw pass arrow from passer to receiver
    try:
        passer_row = df_players[df_players['is_passer']].iloc[0]
        receiver_row = df_players[df_players['player_id'] == receiver_id].iloc[0]

        plt.arrow(
            passer_row['posX'], passer_row['posY'],
            receiver_row['posX'] - passer_row['posX'],
            receiver_row['posY'] - passer_row['posY'],
            width=0.3, head_width=2.2, length_includes_head=True, color='red', alpha=0.9
        )
    except:
        pass

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
row = df_passes.iloc[20]  #   df_passes_cleaned.iloc[500]
pass_row, frame_players = extract_frame_players_and_roles(df_coordinates, row)
draw_frame_players(frame_players, pass_row)


In [ ]:
from collections import Counter
error_counter = Counter()



# NOTE: comment translated/cleaned for English-only repository.
save_dir = r"C:\Users\Adolph\graduation thesis\image"
os.makedirs(save_dir, exist_ok=True)
num_samples = 5000  #        

# NOTE: comment translated/cleaned for English-only repository.
df = df.copy()
df['is_valuable'] = (df['window_score'] > 0.5).astype(int)

# NOTE: comment translated/cleaned for English-only repository.
df_positive = df[df['is_valuable'] == 1]
df_negative = df[df['is_valuable'] == 0].sample(n=max(0, num_samples - len(df_positive)), random_state=42)
df_selected = pd.concat([df_positive, df_negative]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# NOTE: comment translated/cleaned for English-only repository.
def draw_pass_frame_as_image(df_players, pass_row, save_path):
    plt.figure(figsize=(10, 6.5))
    plt.xlim(0, 105)
    plt.ylim(0, 68)
    plt.gca().set_facecolor('lightgreen')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

    receiver_id = pass_row.get('receivedPlayerId', None)
    passer = df_players[df_players['is_passer']]
    plt.scatter(passer['posX'], passer['posY'], color='blue', s=100, label='Passer')

    teammates = df_players[df_players['is_teammate'] & (df_players['player_id'] != receiver_id)]
    plt.scatter(teammates['posX'], teammates['posY'], color='white', s=60, label='Teammates')

    receiver = df_players[df_players['player_id'] == receiver_id]
    if not receiver.empty:
        plt.scatter(receiver['posX'], receiver['posY'], color='yellow', edgecolors='black', s=100, label='Receiver', zorder=5)

    opponents = df_players[df_players['is_opponent']]
    plt.scatter(opponents['posX'], opponents['posY'], color='black', s=60, label='Opponents')

    for _, row in df_players.iterrows():
        if pd.notna(row['player_id']):
            plt.text(row['posX'], row['posY'] + 1, str(int(row['player_id'])), fontsize=8, ha='center')

    try:
        passer_row = passer.iloc[0]
        receiver_row = receiver.iloc[0]
        plt.arrow(
            passer_row['posX'], passer_row['posY'],
            receiver_row['posX'] - passer_row['posX'],
            receiver_row['posY'] - passer_row['posY'],
            width=0.3, head_width=2.2, length_includes_head=True, color='red', alpha=0.9
        )
    except:
        pass

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# NOTE: comment translated/cleaned for English-only repository.
label_rows = []
count = 0

for idx, pass_row in df_selected.iterrows():
    ts = pass_row['timestamp']
    if pd.isna(ts):
        error_counter["timestamp NaN"] += 1
        continue
    try:
        pass_row_out, df_players = extract_frame_players_and_roles(df_coordinates, pass_row)

        img_name = f"pass_{idx:04d}.png"
        img_path = os.path.join(save_dir, img_name)
        draw_pass_frame_as_image(df_players, pass_row_out, img_path)

        label_rows.append({
            'filename': img_name,
            'is_valuable': int(pass_row['is_valuable'])
        })
        count += 1

    except Exception as e:
        error_counter[str(e)] += 1
        continue

# NOTE: comment translated/cleaned for English-only repository.
df_labels = pd.DataFrame(label_rows)
csv_path = os.path.join(save_dir, "pass_labels_valuable.csv")
df_labels.to_csv(csv_path, index=False)

print(f"Images generated: {count}")
print(f"Label CSV saved to: {csv_path}")
print(df_labels['is_valuable'].value_counts())
print(df_labels.head())
print("Error Summary:")
for err, count in error_counter.most_common():
    print(f"{err}: {count}")

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# NOTE: comment translated/cleaned for English-only repository.
image_dir = r"C:\Users\Adolph\graduation thesis\image"
label_path = os.path.join(image_dir, "pass_labels_valuable.csv")
df = pd.read_csv(label_path)

# NOTE: comment translated/cleaned for English-only repository.
df = df.dropna(subset=['is_valuable'])
df['is_valuable'] = df['is_valuable'].astype(int)

# NOTE: comment translated/cleaned for English-only repository.
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_valuable'])

# NOTE: comment translated/cleaned for English-only repository.
class ImageDataset(Sequence):
    def __init__(self, df, batch_size=32, image_size=(224, 224), shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.image_size = image_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        images, labels = [], []
        for _, row in batch_df.iterrows():
            img_path = os.path.join(image_dir, row['filename'])
            img = load_img(img_path, target_size=self.image_size)
            img_array = preprocess_input(img_to_array(img))  #    ResNet50    
            images.append(img_array)
            labels.append(row['is_valuable'])
        return np.array(images), np.array(labels, dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# NOTE: comment translated/cleaned for English-only repository.
train_gen = ImageDataset(train_df)
val_gen = ImageDataset(val_df, shuffle=False)

# NOTE: comment translated/cleaned for English-only repository.
def build_resnet_model(input_shape=(224, 224, 3)):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  #       
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs=base_model.input, outputs=output)

model = build_resnet_model()
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss=losses.BinaryCrossentropy(),
    metrics=[
        metrics.BinaryAccuracy(),
        metrics.Precision(),
        metrics.Recall(),
        metrics.AUC()
    ]
)

# NOTE: comment translated/cleaned for English-only repository.
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint("best_resnet_model.h5", save_best_only=True)
]

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight={0: 1.0, 1: 3.0},
    callbacks=callbacks,
    verbose=1
)

# NOTE: comment translated/cleaned for English-only repository.
model.save("resnet50_cnn_is_valuable_final.h5")


In [ ]:
def extract_frame_players_and_roles(coordinates_df, row):
    timestamp = row['timestamp']
    passer_id = row['passedPlayerId']
    team_id = row['passedPlayerTeamId']

    frame = coordinates_df[coordinates_df['timestamp'] == timestamp]
    if frame.empty or pd.isna(timestamp):
        raise ValueError(f"No coordinate data found for timestamp {timestamp}")
    frame = frame.iloc[0]

    # Extract all players with valid IDs and positions
    player_rows = []
    for i in range(40):
        pid = frame.get(f'playerId{i}', -100)
        x = frame.get(f'posX{i}', -100)
        y = frame.get(f'posY{i}', -100)
        if pid == -100 or x == -100 or y == -100:
            continue
        player_rows.append({'player_index': i, 'player_id': pid, 'posX': x, 'posY': y})

    df_players = pd.DataFrame(player_rows)

    # Retrieve teammate and opponent IDs from pass record
    teammate_ids = [row.get(f'teamMatePlayerId{i}') for i in range(1, 11)]
    opponent_ids = [row.get(f'opponentPlayerId{i}') for i in range(1, 12)]

    # Assign roles
    df_players['is_passer'] = df_players['player_id'] == passer_id
    df_players['is_teammate'] = df_players['player_id'].isin(teammate_ids)
    df_players['is_opponent'] = df_players['player_id'].isin(opponent_ids)

    return row, df_players


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrow
import numpy as np

def draw_frame_players_beautiful(df_players, pass_row):
    fig, ax = plt.subplots(figsize=(10, 6.5))
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 68)
    ax.set_facecolor('#99ff99')  #           
    ax.set_title("Tactical Context of the Pass", fontsize=16, fontweight='bold')

# NOTE: comment translated/cleaned for English-only repository.
    passer = df_players[df_players['is_passer']]
    x0, y0 = passer['posX'].values[0], passer['posY'].values[0]
    if pass_row.get('isSucceeded') == 0:
        x1 = pass_row.get('receivedPlayerPosXh', -1)
        y1 = pass_row.get('receivedPlayerPosYh', -1)
    else:
        x1 = pass_row.get('receivedPlayerPosX', -1)
        y1 = pass_row.get('receivedPlayerPosY', -1)

# NOTE: comment translated/cleaned for English-only repository.
    ax.scatter(x0, y0, color='#1e88e5', s=120, label='Passer', zorder=5)
    ax.scatter(df_players[df_players['is_teammate']]['posX'],
               df_players[df_players['is_teammate']]['posY'],
               color='white', edgecolors='gray', s=80, label='Teammates', zorder=3)
    ax.scatter(df_players[df_players['is_opponent']]['posX'],
               df_players[df_players['is_opponent']]['posY'],
               color='black', s=80, label='Opponents', zorder=3)
    if x1 > 0 and y1 > 0:
        ax.scatter(x1, y1, color='gold', edgecolors='black', s=120, label='Receiver', zorder=6)

# NOTE: comment translated/cleaned for English-only repository.
        ax.add_patch(FancyArrow(x0, y0, x1 - x0, y1 - y0,
                                width=0.3, head_width=2.2, head_length=2,
                                length_includes_head=True, color='crimson', alpha=0.85, zorder=4))

# NOTE: comment translated/cleaned for English-only repository.
        dx, dy = x1 - x0, y1 - y0
        norm = np.sqrt(dx**2 + dy**2)
        if norm > 0:
            dx, dy = dx / norm, dy / norm
            normal = np.array([-dy, dx])
            for _, row in df_players[df_players['is_opponent']].iterrows():
                ox, oy = row['posX'], row['posY']
                vec = np.array([ox - x0, oy - y0])
                proj_len = np.dot(vec, [dx, dy])
                dist_to_path = abs(np.dot(vec, normal))
                if 0 <= proj_len <= norm and dist_to_path <= 2:
                    ax.scatter(ox, oy, s=200, facecolors='none', edgecolors='purple',
                               linewidths=2, label='In Path' if 'In Path' not in ax.get_legend_handles_labels()[1] else "", zorder=6)

# NOTE: comment translated/cleaned for English-only repository.
        vec_pass = np.array([x1 - x0, y1 - y0])
        norm_pass = np.linalg.norm(vec_pass)
        if norm_pass > 0:
            vec_pass = vec_pass / norm_pass
            min_dist = float('inf')
            nearest_vec = None
            for _, row in df_players[df_players['is_opponent']].iterrows():
                dxo, dyo = row['posX'] - x0, row['posY'] - y0
                dist = np.sqrt(dxo**2 + dyo**2)
                if dist > 0 and dist < min_dist:
                    min_dist = dist
                    nearest_vec = np.array([dxo, dyo]) / dist
            if nearest_vec is not None:
                ax.add_patch(FancyArrow(x0, y0, nearest_vec[0]*15, nearest_vec[1]*15,
                                        width=0.2, head_width=1.5, color='royalblue', alpha=0.6,
                                        label='Angle to Nearest Opponent'))

# NOTE: comment translated/cleaned for English-only repository.
    pressure_level = pass_row.get("pressure_level_num", -1)
    pressure_colors = {0: "#bbdefb", 1: "#ffe082", 2: "#ef9a9a"}
    pressure_labels = {0: "Low Pressure", 1: "Medium Pressure", 2: "High Pressure"}
    color = pressure_colors.get(pressure_level, "gray")
    label = pressure_labels.get(pressure_level, "Pressure Area")
    ax.add_patch(Circle((x0, y0), radius=10, color=color, alpha=0.3, label=label))

    ax.set_xlabel("Field X", fontsize=12)
    ax.set_ylabel("Field Y", fontsize=12)
    ax.legend(loc='upper right', frameon=True, framealpha=0.95, edgecolor='gray', fancybox=True)
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


In [ ]:
pass_row, frame_players = extract_frame_players_and_roles(df_coordinates, row)
draw_frame_players_beautiful(frame_players, pass_row)
row = df_passes.iloc[180]


In [ ]:
def extract_frame_players_and_roles(coordinates_df, passes_df, row_idx):
    # Extract pass event row
    row = passes_df.loc[row_idx]
    timestamp = row['normal_time']
    passer_id = row['passedPlayerId']
    team_id = row['passedPlayerTeamId']

    # Get corresponding coordinate frame at the same timestamp
    frame = coordinates_df[coordinates_df['timestamp'] == timestamp]
    if frame.empty:
        raise ValueError(f"No coordinate data found for timestamp {timestamp}")
    frame = frame.iloc[0]

    # Extract all players with valid IDs and positions
    player_rows = []
    for i in range(40):
        pid = frame.get(f'playerId{i}', -100)
        x = frame.get(f'posX{i}', -100)
        y = frame.get(f'posY{i}', -100)
        if pid == -100 or x == -100 or y == -100:
            continue
        player_rows.append({'player_index': i, 'player_id': pid, 'posX': x, 'posY': y})

    df_players = pd.DataFrame(player_rows)

    # Retrieve teammate and opponent IDs from pass record
    teammate_ids = [row.get(f'teamMatePlayerId{i}') for i in range(1, 11)]
    opponent_ids = [row.get(f'opponentPlayerId{i}') for i in range(1, 12)]

    # Assign roles
    df_players['is_passer'] = df_players['player_id'] == passer_id
    df_players['is_teammate'] = df_players['player_id'].isin(teammate_ids)
    df_players['is_opponent'] = df_players['player_id'].isin(opponent_ids)

    return row, df_players


In [ ]:
def draw_frame_players(df_players, pass_row):
    # Set up pitch display
    plt.figure(figsize=(10, 6.5))
    plt.xlim(0, 105)
    plt.ylim(0, 68)
    plt.gca().set_facecolor('lightgreen')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

    receiver_id = pass_row.get('receivedPlayerId', None)

    # Draw passer
    passer = df_players[df_players['is_passer']]
    plt.scatter(passer['posX'], passer['posY'], color='blue', s=100, label='Passer')

    # Draw teammates (excluding receiver)
    teammates = df_players[df_players['is_teammate'] & (df_players['player_id'] != receiver_id)]
    plt.scatter(teammates['posX'], teammates['posY'], color='white', s=60, label='Teammates')

    # Draw receiver
    receiver = df_players[df_players['player_id'] == receiver_id]
    if not receiver.empty:
        plt.scatter(receiver['posX'], receiver['posY'], color='yellow', edgecolors='black', s=100, label='Receiver', zorder=5)

    # Draw opponents
    opponents = df_players[df_players['is_opponent']]
    plt.scatter(opponents['posX'], opponents['posY'], color='black', s=60, label='Opponents')

    # Annotate player IDs
    for _, row in df_players.iterrows():
        if pd.notna(row['player_id']):
            plt.text(row['posX'], row['posY'] + 1, str(int(row['player_id'])), fontsize=8, ha='center')

    # Draw pass arrow from passer to receiver
    try:
        passer_row = df_players[df_players['is_passer']].iloc[0]
        receiver_row = df_players[df_players['player_id'] == receiver_id].iloc[0]

        plt.arrow(
            passer_row['posX'], passer_row['posY'],
            receiver_row['posX'] - passer_row['posX'],
            receiver_row['posY'] - passer_row['posY'],
            width=0.3, head_width=2.2, length_includes_head=True, color='red', alpha=0.9
        )
    except:
        pass

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
pass_row, frame_players = extract_frame_players_and_roles(df_coordinates, df_passes, 200)
draw_frame_players(frame_players, pass_row)


In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, 15))  #      14   epoch    


plt.figure(figsize=(10, 6))
plt.plot(epochs, val_auc, label='Val AUC', marker='o')
plt.plot(epochs, val_accuracy, label='Val Accuracy', marker='s')
plt.plot(epochs, val_precision, label='Val Precision', marker='^')
plt.plot(epochs, val_recall, label='Val Recall', marker='x')

plt.xlabel('Epoch')
plt.ylabel('Metric Value')
plt.title('CNN Validation Metrics Across Epochs')
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("cnn_validation_metrics.png", dpi=300)
plt.show()
